# Process Mining with Python and PM4Py
## A fictional investment-banking booking process — Learner edition

**Duration:** 60 minutes &nbsp; | &nbsp; **Level:** Intermediate Python, new to process mining

> **Fictional-data notice:** Every trade, counterparty, resource, and event in this workshop is synthetic.
> An anomaly is a reason to investigate—not proof of error or misconduct.

## Learning objectives and agenda

By the end, you will be able to:

1. explain what process mining adds to ordinary reporting;
2. recognize and prepare a case-based event log;
3. discover and visualize control flow and performance with PM4Py;
4. find suspicious cases with rare variants, duration, and conformance diagnostics.

| Time | Topic |
|---:|---|
| 0–8 min | What process mining is |
| 8–20 min | Load and prepare an event log |
| 20–40 min | Discover and visualize the process |
| 40–56 min | Find and explain outliers |
| 56–60 min | Recap and next steps |

In [ ]:
from pathlib import Path
import shutil
import tempfile

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pm4py
from IPython.display import SVG, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

print("PM4Py version:", pm4py.__version__)
print("Graphviz dot available:", shutil.which("dot") is not None)

# 1. What is process mining? (0–8 minutes)

A business process is not just a flowchart: it is what cases actually did over time. **Process mining**
reconstructs and measures that behavior from event data.

Every useful event log needs three essential ideas:

- **Case:** one process instance, such as one booking ID.
- **Activity:** a step performed for the case, such as `Compliance Check`.
- **Timestamp:** when that step occurred.

Optional attributes—resource, desk, product, amount, or counterparty tier—help explain *why* behavior differs.

The three common process-mining questions are:

- **Discovery:** What process does the data reveal?
- **Performance:** Where are delays, queues, or rework?
- **Conformance:** Which cases deviate from expected behavior?

Traditional BI usually aggregates rows into totals. Process mining retains the **sequence within each case**,
which exposes loops, skipped controls, premature steps, and alternative paths.

### Quick check: which table is an event log?

**A.** One row per desk with monthly trade count and average value.  
**B.** One row per booking event with booking ID, activity, and timestamp.  
**C.** One row per employee with team and job title.

<details><summary>Reveal the answer</summary>

**B.** It preserves the case, activity, and time needed to reconstruct each trace. A and C may enrich an
analysis, but cannot reconstruct a process by themselves.
</details>

### Scenario

A fictional investment bank wants to understand its post-trade booking process. A normal booking may include:

`Trade Captured → Validate Economics → Enrich Counterparty → Compliance Check → Credit Check →`
`[Supervisor Approval for high-value trades] → Confirm Booking → Post to Ledger → Booking Complete`

Some legitimate cases require one amendment and revalidation. We will inspect 120 historical baseline cases
and 40 later monitoring cases.

**Predict before coding:** Where would you expect rework, delays, or control failures to appear?

# 2. Load and prepare the event log (8–20 minutes)

The CSV deliberately contains no anomaly label. In real work, the point is to find behavior worth reviewing
before an investigator confirms what happened.

### Your turn 1 — load and inspect

Load the CSV, parse the timestamp column, and show the first five rows.

In [ ]:
DATA_PATH = Path("data/investment_banking_booking_log.csv")

# TODO: load DATA_PATH with pandas.
# Hint: use parse_dates=[TIMESTAMP].
raw_log = None
raw_log

### Data contract

| Column | Meaning |
|---|---|
| `case:concept:name` | Booking/case identifier |
| `concept:name` | Activity name |
| `time:timestamp` | UTC event timestamp |
| `org:resource` | Fictional person or system performing the event |
| `booking_period` | `baseline` or `monitoring` cohort |
| `desk`, `product`, `currency` | Trade dimensions |
| `counterparty_tier`, `notional_usd` | Case context repeated on each event |

### Your turn 2 — validate the minimum event-log quality

Calculate row count, case count, activity count, missing values in required columns, and whether timestamps
increase within every case.

In [ ]:
required = [CASE_ID, ACTIVITY, TIMESTAMP]

# TODO: populate this dictionary.
quality = {
    "events": None,
    "cases": None,
    "activities": None,
    "missing_required_values": None,
    "all_cases_time_ordered": None,
}
pd.Series(quality, name="value")

PM4Py accepts a pandas DataFrame when the case, activity, and timestamp keys are supplied. Formatting makes
the types and ordering explicit. We retain business attributes for later slicing and interpretation.

In [ ]:
# TODO: use pm4py.format_dataframe and sort by case and timestamp.
log = None
baseline = None
monitoring = None

### Your turn 3 — summarize cases and durations

Build one row per case with start time, end time, event count, cohort, and duration in minutes. Then compare
median duration by cohort.

In [ ]:
# TODO: group by CASE_ID and aggregate start/end/event count/cohort.
case_summary = None

# 3. Discover and visualize the process (20–40 minutes)

A **trace** is the ordered activity sequence of one case. A **variant** is a distinct trace shared by one or
more cases. Variant frequency gives a compact view of the process's dominant and unusual routes.

### Your turn 4 — extract variants

Create an activity tuple for every case, count the tuples, and display the ten most frequent variants.

In [ ]:
# TODO: create variant_by_case and variant_counts.
variant_by_case = None
variant_counts = None

**Interpretation prompt:** Is the most common route the only valid route? Which optional step or loop explains
the main alternatives? A rare route is a useful signal, but rarity alone is not a control failure.

A **Directly-Follows Graph (DFG)** counts adjacent activity pairs across cases. A frequency DFG shows what
happens often; a performance DFG shows elapsed time between adjacent activities.

In [ ]:
def draw_dfg(dfg, start_activities, end_activities, title, performance=False, max_edges=18):
    """Render through PM4Py/Graphviz when available, otherwise use a notebook-safe fallback."""
    if shutil.which("dot"):
        output = Path(tempfile.mkdtemp()) / "dfg.svg"
        if performance:
            pm4py.save_vis_performance_dfg(
                dfg, start_activities, end_activities, str(output),
                rankdir="LR", graph_title=title, max_num_edges=max_edges,
            )
        else:
            pm4py.save_vis_dfg(
                dfg, start_activities, end_activities, str(output),
                rankdir="LR", graph_title=title, max_num_edges=max_edges,
            )
        display(SVG(filename=str(output)))
        return

    def metric_value(value):
        if isinstance(value, dict):
            return float(value.get("mean", next(iter(value.values()))))
        return float(value)

    strongest = sorted(
        dfg.items(), key=lambda item: metric_value(item[1]), reverse=True
    )[:max_edges]
    graph = nx.DiGraph()
    for (source, target), value in strongest:
        graph.add_edge(source, target, weight=metric_value(value))
    process_positions = {
        "Trade Captured": (0, 0),
        "Validate Economics": (1, 0),
        "Amend Booking": (1, 1),
        "Enrich Counterparty": (2, 0),
        "Compliance Check": (3, 0),
        "Manual Override": (3.5, 1),
        "Credit Check": (4, 0),
        "Supervisor Approval": (5, 1),
        "Confirm Booking": (5, 0),
        "Post to Ledger": (6, 0),
        "Booking Complete": (7, 0),
    }
    positions = {
        node: process_positions.get(node, (index, -1))
        for index, node in enumerate(graph.nodes)
    }
    plt.figure(figsize=(15, 5.5))
    nx.draw_networkx(
        graph, positions, node_color="#dbeafe", edge_color="#64748b",
        node_size=2300, font_size=8, arrowsize=18,
        connectionstyle="arc3,rad=0.06",
    )
    labels = {
        edge: (
            f"{metric_value(value) / 60:.1f}m"
            if performance else f"{int(metric_value(value))}"
        )
        for edge, value in strongest
    }
    nx.draw_networkx_edge_labels(
        graph, positions, edge_labels=labels, font_size=7,
        connectionstyle="arc3,rad=0.06",
    )
    plt.xlim(-0.6, 7.6)
    plt.ylim(-0.65, 1.65)
    plt.title(title + " (NetworkX fallback)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

### Your turn 5 — discover the frequency DFG

Use `pm4py.discover_dfg` on the full log, then visualize it. Before running the cell, predict which edge will
have the highest frequency.

In [ ]:
# TODO: discover dfg, start_activities, and end_activities with PM4Py.
dfg = start_activities = end_activities = None
# draw_dfg(dfg, start_activities, end_activities, "Booking process — frequency")

### Your turn 6 — discover the performance DFG

Repeat the discovery with `pm4py.discover_performance_dfg`. Edge labels are mean elapsed times. Which edge
looks slow, and is it slow for every case or only a few?

In [ ]:
# TODO: discover and draw the performance DFG.
performance_dfg = performance_starts = performance_ends = None

A DFG is descriptive. For formal conformance diagnostics, we discover a Petri net from the known-clean
baseline cohort using the **Inductive Miner**. The same discovery also produces a readable process tree.

### Your turn 7 — discover a reference model

In [ ]:
# TODO: discover a process tree and Petri net from baseline.
process_tree = None
net = initial_marking = final_marking = None

# 4. Find outliers with three complementary signals (40–56 minutes)

No single anomaly rule is sufficient:

- **Rare variant:** good for unusual routes, but can flag legitimate rare work.
- **Long duration:** good for delay, but misses fast control violations.
- **Low conformance fitness:** good for unexpected order or activities, but may accept unusual repetitions if
  the reference model contains a loop.

We will flag a monitoring case when **any** signal fires, then preserve the reasons for human review.

### Signal 1 — rare monitoring variants

For this small teaching log, define rare as at most three monitoring cases. In production this threshold
should be calibrated against volume, seasonality, and investigation capacity.

In [ ]:
# TODO: compute variant frequency in monitoring and flag frequency <= 3.
monitoring_variants = None

### Signal 2 — unusually long duration

Learn a robust upper bound from baseline duration: `Q3 + 1.5 × IQR`. This avoids using planted labels and is
less sensitive to a few large values than mean plus standard deviation.

In [ ]:
# TODO: calculate the baseline IQR threshold and flag monitoring durations above it.
duration_upper_bound = None
duration_signals = None

### Signal 3 — conformance fitness

An alignment compares an observed trace with the reference Petri net. A fitness of `1.0` means the trace can
be replayed perfectly. Lower fitness indicates moves present only in the log or only in the model.

**Prediction:** Will an excessive number of repetitions always have low fitness when the model contains a loop?

In [ ]:
# TODO: align the monitoring EventLog to the reference Petri net.
alignment_scores = None

### Your turn 8 — combine and rank the evidence

Join the three case-level signals. Create a readable `anomaly_reasons` field and rank flagged cases by number
of signals, then by lowest fitness and longest duration.

In [ ]:
# TODO: merge the three signal tables and build is_anomaly/anomaly_reasons.
ranked_outliers = None

### Investigate before judging

Pick two flagged booking IDs and inspect their ordered traces. Explain in plain business language what looks
unusual. Then inspect one unflagged case as a comparison.

In [ ]:
# TODO: choose case IDs and print their ordered activity/timestamp/resource history.
cases_to_review = []
log.loc[log[CASE_ID].isin(cases_to_review), [CASE_ID, ACTIVITY, TIMESTAMP, "org:resource"]]

### Validate against planted truth

During real investigations, ground truth comes from source records and domain experts. For this synthetic
lesson only, the instructor answer key lets us measure precision, recall, and coverage by anomaly type.

In [ ]:
# The learner edition intentionally does not load or expose the planted labels.
# After ranking cases, compare your explanations with the instructor or solved notebook.

# 5. Recap and next steps (56–60 minutes)

The reusable workflow is:

1. **Frame the case and event semantics.** Bad identifiers or timestamps produce misleading models.
2. **Explore variants and performance.** Understand dominant paths before labeling deviations.
3. **Discover a reference model from trustworthy behavior.** Avoid training blindly on known incidents.
4. **Combine signals.** Rarity, time, and conformance reveal different failure modes.
5. **Investigate with context.** Thresholds prioritize work; they do not replace business judgment.

**Exit question:** Which signal found something the other two could miss, and what false positive could it create?

Optional extensions:

- compare desks, products, counterparties, or resources;
- use PM4Py temporal profiles for activity-to-activity timing deviations;
- monitor model fitness over time for process drift;
- export XES and connect findings to a case-management workflow.